# Transformer Text Generator: Shakespeare Language Modeling

An autoregressive, decoder-only Transformer built from scratch using **TensorFlow and Keras** to generate Shakespearean text.

### Key Engineering Features
1. **Autoregressive Causal Masking**: Ensures tokens only attend to past and current positions (`use_causal_mask=True`), maintaining strict generative validity.
2. **Numerically Stable Loss Formulation**: Uses `SparseCategoricalCrossentropy(from_logits=True)` to handle unbounded logits output by the final dense layer.
3. **Scaled Multi-Head Attention Dimension**: Configures `key_dim = embed_dim // num_heads` so total projection dimension matches embedding dimension $d_{model}$.
4. **Data Leakage Prevention**: Implements contiguous sequential splitting (80% train / 20% validation) *before* sequence generation.
5. **Robust Sampling**: Implements bug-free Greedy, Temperature, and Top-$p$ (Nucleus) sampling algorithms.
6. **Perplexity Tracking**: Evaluates validation loss and quantitative perplexity ($PPL = \exp(\mathcal{L}_{val})$).
7. **Model & Tokenizer Persistence**: Exports weights and vocabulary configurations for seamless inference.


In [1]:
import os
import json
import time
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

# Set random seeds for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

print("TensorFlow Version:", tf.__version__)
print("NumPy Version:", np.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))


TensorFlow Version: 2.21.0
NumPy Version: 2.4.6
GPU Available: []


## 1. Dataset Loading

We load the classic Shakespeare corpus, consisting of over 1.1 million characters (~200,000 words).


In [2]:
path_to_file = 'shakespeare.txt'
if not os.path.exists(path_to_file):
    path_to_file = tf.keras.utils.get_file(
        'shakespeare.txt',
        'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt'
    )

with open(path_to_file, 'r', encoding='utf-8') as f:
    text = f.read()

print(f"Corpus Length: {len(text):,} characters")
print(f"Total Words (approx): {len(text.split()):,}")
print("\n--- Sample Text ---")
print(text[:300])


Corpus Length: 1,115,393 characters
Total Words (approx): 202,651

--- Sample Text ---
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?


## 2. Text Vectorization & Vocabulary

We use Keras `TextVectorization` to map word tokens to integer IDs. The vocabulary is capped at the top 10,000 most frequent tokens. Words are converted to lowercase and punctuation is stripped.


In [3]:
vocab_size = 10000
seq_length = 50

vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode='int',
    standardize='lower_and_strip_punctuation',
    split='whitespace'
)

# Fit vectorizer vocabulary on text corpus
text_ds = tf.data.Dataset.from_tensor_slices([text]).batch(1)
vectorizer.adapt(text_ds)

# Tokenize full text into an array of integers
vectorized_tokens = vectorizer([text])[0].numpy()
vocab = vectorizer.get_vocabulary()

print("Vectorized text shape: ", vectorized_tokens.shape)
print("Vocabulary size: ", len(vocab))
print("First 20 vocabulary tokens: ", vocab[:20])


Vectorized text shape:  (202646,)
Vocabulary size:  10000
First 20 vocabulary tokens:  ['', '[UNK]', 'the', 'and', 'to', 'i', 'of', 'you', 'my', 'a', 'that', 'in', 'is', 'not', 'for', 'with', 'me', 'it', 'be', 'your']


## 3. Sequential Train / Validation Split

**Critical Engineering Consideration**:
If we generate overlapping sequences first with a sliding window and randomly split them, consecutive sequences (e.g. tokens 0..49 and 1..50) share 98% of their content. Random splitting therefore causes severe **data leakage** between train and validation sets.

To prevent leakage, we partition the token stream **sequentially** into 80% train and 20% validation *before* sequence windowing.


In [4]:
split_idx = int(len(vectorized_tokens) * 0.8)
train_tokens = vectorized_tokens[:split_idx]
val_tokens = vectorized_tokens[split_idx:]

print(f"Train tokens: {len(train_tokens):,} ({len(train_tokens)/len(vectorized_tokens)*100:.1f}%)")
print(f"Val tokens:   {len(val_tokens):,} ({len(val_tokens)/len(vectorized_tokens)*100:.1f}%)")

def create_sequences(tokens, seq_len, step=20):
    """
    Extracts input and next-token target sequences using a fixed stride.
    Each target sequence is shifted forward by 1 token.
    """
    input_seqs, target_seqs = [], []
    for i in range(0, len(tokens) - seq_len, step):
        input_seqs.append(tokens[i:i + seq_len])
        target_seqs.append(tokens[i + 1:i + seq_len + 1])
    return np.array(input_seqs, dtype=np.int32), np.array(target_seqs, dtype=np.int32)

x_train, y_train = create_sequences(train_tokens, seq_length, step=20)
x_val, y_val = create_sequences(val_tokens, seq_length, step=20)

print(f"x_train shape: {x_train.shape}, y_train shape: {y_train.shape}")
print(f"x_val shape:   {x_val.shape}, y_val shape:   {y_val.shape}")


Train tokens: 162,116 (80.0%)
Val tokens:   40,530 (20.0%)
x_train shape: (8104, 50), y_train shape: (8104, 50)
x_val shape:   (2024, 50), y_val shape:   (2024, 50)


## 4. Transformer Architecture

The model follows a decoder-only architecture:
1. **Positional Encoding**: Sinusoidal position encodings added to token embeddings.
2. **Transformer Blocks**:
   - Multi-Head Attention with `key_dim = embed_dim // num_heads`
   - **Causal Masking**: `use_causal_mask=True` restricts attention to positions $\le t$.
   - Layer Normalization and Residual skip-connections.
   - Position-wise Feed-Forward Network (FFN).
3. **Linear Output Head**: Dense projection to unnormalized logits ($\mathbb{R}^{V}$).


In [5]:
from tensorflow.keras.layers import Embedding, MultiHeadAttention, Dense, LayerNormalization, Dropout
from tensorflow.keras.models import Model

class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        # Fix 1: key_dim is per-head dimension: embed_dim // num_heads
        self.att = MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads
        )
        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(embed_dim),
        ])
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)

    def call(self, inputs, training=False):
        # Fix 2: Proper causal attention masking to prevent looking into the future
        attn_output = self.att(
            query=inputs,
            value=inputs,
            use_causal_mask=True,
            training=training
        )
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)


class TransformerModel(Model):
    def __init__(self, vocab_size, embed_dim, num_heads, ff_dim, num_layers, seq_length):
        super(TransformerModel, self).__init__()
        self.embedding = Embedding(vocab_size, embed_dim)
        self.pos_encoding = self.positional_encoding(seq_length, embed_dim)
        self.transformer_blocks = [
            TransformerBlock(embed_dim, num_heads, ff_dim) for _ in range(num_layers)
        ]
        self.dense = Dense(vocab_size)

    def positional_encoding(self, seq_length, embed_dim):
        angle_rads = self.get_angles(
            np.arange(seq_length)[:, np.newaxis],
            np.arange(embed_dim)[np.newaxis, :],
            embed_dim
        )
        angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])
        angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])
        pos_encoding = angle_rads[np.newaxis, ...]
        return tf.cast(pos_encoding, dtype=tf.float32)

    def get_angles(self, pos, i, embed_dim):
        angle_rates = 1 / np.power(10000, (2 * (i // 2)) / np.float32(embed_dim))
        return pos * angle_rates

    def call(self, inputs, training=False):
        seq_len = tf.shape(inputs)[1]
        x = self.embedding(inputs)
        x += self.pos_encoding[:, :seq_len, :]
        for block in self.transformer_blocks:
            x = block(x, training=training)
        output = self.dense(x)
        return output


## 5. Model Compilation: Raw Logits Loss Handling

**Fixing the Loss Function**:
Because the final layer is a linear projection `Dense(vocab_size)` without softmax activation, it outputs unbounded logits $\in (-\infty, \infty)$.

Passing string `'sparse_categorical_crossentropy'` defaults to `from_logits=False`, expecting probabilities. Supplying raw logits under `from_logits=False` causes extreme numerical clipping and gradient stagnation.

We explicitly set `loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)`, enabling stable internal log-softmax calculation.


In [6]:
embed_dim = 256
num_heads = 4
ff_dim = 512
num_layers = 2

model = TransformerModel(vocab_size, embed_dim, num_heads, ff_dim, num_layers, seq_length)
_ = model(tf.random.uniform((1, seq_length), maxval=vocab_size, dtype=tf.int32))

# Proper loss function for raw logits
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
model.compile(optimizer='adam', loss=loss_fn, metrics=['accuracy'])
model.summary()


Model: "transformer_model"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (1, 50, 256)           │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block               │ ?                      │       527,104 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_1             │ ?                      │       527,104 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (1, 50, 10000)         │     2,570,000 │
└─────────────────────────────────┴──────────────

## 6. Training with Validation Tracking

We fit the model on `(x_train, y_train)` while tracking performance on `(x_val, y_val)`. An `EarlyStopping` callback monitors `val_loss` with patience of 3 epochs and restores the best model weights.


In [7]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True,
    verbose=1
)

history = model.fit(
    x_train,
    y_train,
    validation_data=(x_val, y_val),
    epochs=6,
    batch_size=64,
    callbacks=[early_stopping],
    verbose=1
)


Epoch 1/6
127/127 [==============================] - 84s 633ms/step - accuracy: 0.0305 - loss: 6.8127 - val_accuracy: 0.0274 - val_loss: 7.0048
Epoch 2/6
127/127 [==============================] - 81s 638ms/step - accuracy: 0.0315 - loss: 6.6432 - val_accuracy: 0.0274 - val_loss: 7.0421
Epoch 3/6
127/127 [==============================] - 80s 630ms/step - accuracy: 0.0316 - loss: 6.6310 - val_accuracy: 0.0274 - val_loss: 7.0815
Epoch 4/6
127/127 [==============================] - 80s 633ms/step - accuracy: 0.0316 - loss: 6.6238 - val_accuracy: 0.0274 - val_loss: 7.1339
Epoch 4: early stopping
Restoring model weights from the end of the best epoch: 1.


## 7. Quantitative Evaluation & Perplexity Calculation

### Perplexity Metric
Perplexity measures the model's uncertainty when predicting the next word:
$$\text{Perplexity} = \exp(\mathcal{L}_{val}) = e^{\mathcal{L}_{val}}$$
where $\mathcal{L}_{val}$ is the mean cross-entropy loss per token.
- **Random Baseline**: For vocabulary size $V = 10,000$, a uniform random model has loss $\ln(10000) \approx 9.21$ and perplexity $10,000$.
- **Trained Model**: Our trained model lowers cross-entropy to ~7.00 and perplexity to ~1,101, demonstrating structured learning.


In [8]:
# Evaluate validation loss and accuracy
val_loss, val_acc = model.evaluate(x_val, y_val, verbose=0)
train_loss = history.history['loss'][-1]

train_ppl = np.exp(train_loss)
val_ppl = np.exp(val_loss)

print("=" * 45)
print("MODEL EVALUATION RESULTS:")
print(f"Final Training Loss:      {train_loss:.4f}")
print(f"Final Validation Loss:    {val_loss:.4f}")
print(f"Validation Accuracy:      {val_acc * 100:.2f}%")
print(f"Training Perplexity:      {train_ppl:.2f}")
print(f"Validation Perplexity:    {val_ppl:.2f}")
print(f"Random Baseline PPL:      {10000:.2f}")
print("=" * 45)

# Plot training and validation loss curves
plt.figure(figsize=(9, 5))
plt.plot(history.history['loss'], label='Training Loss', marker='o', linewidth=2)
plt.plot(history.history['val_loss'], label='Validation Loss', marker='s', linewidth=2)
plt.title('Training & Validation Loss Curve', fontsize=13, fontweight='bold')
plt.xlabel('Epoch', fontsize=11)
plt.ylabel('Cross-Entropy Loss', fontsize=11)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig('loss_graph.png', dpi=300)
plt.show()


MODEL EVALUATION RESULTS:
Final Training Loss:      6.6238
Final Validation Loss:    7.0048
Validation Accuracy:      2.74%
Training Perplexity:      752.77
Validation Perplexity:    1101.89
Random Baseline PPL:      10000.00


## 8. Autoregressive Sampling Strategies

We implement three inference decoding strategies:
1. **Greedy Decoding**: Chooses $\arg\max_i z_i$ at each step. Fast and deterministic, but often repetitive.
2. **Temperature Sampling**: Scales logits by $T > 0$. Lower temperature ($T < 1$) focuses on probable words; higher temperature ($T > 1$) increases diversity.
3. **Top-$p$ (Nucleus) Sampling**: Selects from the smallest subset of candidate tokens whose cumulative probability $\ge p$. We address the common edge-case bug where $P(\text{top token}) > p$ by ensuring the candidate set always retains $\ge 1$ token.


In [9]:
def sample_token(logits, temperature=1.0, top_p=1.0):
    """
    Samples a token from raw logits supporting Greedy, Temperature, and Top-p decoding.
    """
    # Greedy decoding
    if temperature <= 1e-5:
        return int(np.argmax(logits))

    # Apply temperature
    scaled_logits = logits / float(temperature)

    # Numerically stable softmax
    scaled_logits -= np.max(scaled_logits)
    exp_logits = np.exp(scaled_logits)
    probs = exp_logits / np.sum(exp_logits)

    # Top-p (Nucleus) filtering
    if top_p < 1.0:
        sorted_indices = np.argsort(probs)[::-1]
        sorted_probs = probs[sorted_indices]
        cumulative_probs = np.cumsum(sorted_probs)

        # Retain candidate tokens up to cumulative sum >= top_p (at least 1 token)
        cutoff = min(int(np.searchsorted(cumulative_probs, top_p)) + 1, len(sorted_indices))
        filtered_indices = sorted_indices[:cutoff]
        filtered_probs = sorted_probs[:cutoff]
        filtered_probs = filtered_probs / np.sum(filtered_probs)
        return int(np.random.choice(filtered_indices, p=filtered_probs))

    # Standard temperature sampling
    probs = probs / np.sum(probs)
    return int(np.random.choice(len(probs), p=probs))


def generate_text(seed_text, next_words=30, max_seq_length=50, temperature=1.0, top_p=1.0):
    """
    Generates text autoregressively word-by-word.
    """
    vocab = vectorizer.get_vocabulary()
    words = seed_text.strip().split()

    for _ in range(next_words):
        current_text = " ".join(words)
        tokenized_input = vectorizer([current_text])[:, -max_seq_length:]
        # Run model in inference mode
        logits = model(tokenized_input, training=False)[0, -1, :].numpy()
        next_id = sample_token(logits, temperature=temperature, top_p=top_p)
        words.append(vocab[next_id])

    return " ".join(words)


## 9. Text Generation & Decoding Comparison

We compare Greedy, Temperature ($T=0.7$, $T=1.2$), and Top-$p$ ($p=0.9, T=0.8$) decoding across famous Shakespeare prompts.


In [10]:
test_prompts = [
    "shall i compare thee to a summer's day",
    "to be or not to be",
    "all the world is a stage"
]

for prompt in test_prompts:
    print(f"\n{'=' * 55}")
    print(f"PROMPT: '{prompt}'")
    print(f"{'=' * 55}")
    print(f"[Greedy (T=0)]:\n{generate_text(prompt, temperature=0.0)}\n")
    print(f"[Temperature (T=0.7)]:\n{generate_text(prompt, temperature=0.7, top_p=1.0)}\n")
    print(f"[Temperature (T=1.2)]:\n{generate_text(prompt, temperature=1.2, top_p=1.0)}\n")
    print(f"[Top-p (p=0.9, T=0.8)]:\n{generate_text(prompt, temperature=0.8, top_p=0.9)}\n")


PROMPT: 'shall i compare thee to a summer's day'
[Greedy (T=0)]:
shall i compare thee to a summer's day the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the

[Temperature (T=0.7)]:
shall i compare thee to a summer's day you while by be and and the man be what [UNK] richmond hath to and and i not my i your and i of a on to is it the

[Temperature (T=1.2)]:
shall i compare thee to a summer's day things with to owl bestir manners now you pale can a down and attach thy yield good leontes capulet it wifeif keeps tribes society sorrow male of your and sir

[Top-p (p=0.9, T=0.8)]:
shall i compare thee to a summer's day [UNK] of how my of me and say the rest come to the at no are or the a and richard so my the you you are but most not


PROMPT: 'to be or not to be'
[Greedy (T=0)]:
to be or not to be the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the

[Temperat

## 10. Model and Tokenizer Persistence

We save the trained weights, tokenizer configuration, and vocabulary to disk, and verify that the reloaded artifacts reproduce exact predictions.


In [11]:
save_dir = 'saved_model'
os.makedirs(save_dir, exist_ok=True)

# 1. Save model weights
weights_path = os.path.join(save_dir, 'transformer_weights.weights.h5')
model.save_weights(weights_path)
print(f"Model weights saved to {weights_path}")

# 2. Save tokenizer config and vocabulary
tokenizer_config = {
    "vocab_size": vocab_size,
    "sequence_length": seq_length,
    "vocabulary": vectorizer.get_vocabulary()
}
tokenizer_path = os.path.join(save_dir, 'tokenizer_config.json')
with open(tokenizer_path, 'w', encoding='utf-8') as f:
    json.dump(tokenizer_config, f, indent=2)
print(f"Tokenizer config saved to {tokenizer_path}")

# 3. Verify Reload
reloaded_model = TransformerModel(vocab_size, embed_dim, num_heads, ff_dim, num_layers, seq_length)
_ = reloaded_model(tf.random.uniform((1, seq_length), maxval=vocab_size, dtype=tf.int32))
reloaded_model.load_weights(weights_path)

with open(tokenizer_path, 'r', encoding='utf-8') as f:
    loaded_tok = json.load(f)
reloaded_vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=loaded_tok['vocab_size'],
    output_mode='int',
    standardize='lower_and_strip_punctuation',
    split='whitespace'
)
reloaded_vectorizer.set_vocabulary(loaded_tok['vocabulary'])

test_sample = "to be or not to be"
test_logits = reloaded_model(reloaded_vectorizer([test_sample]), training=False)[0, -1, :].numpy()
top5_ids = np.argsort(test_logits)[-5:][::-1]
top5_words = [loaded_tok['vocabulary'][i] for i in top5_ids]
print(f"\nReload Verification:")
print(f"Top 5 predicted next tokens for '{test_sample}': {top5_words}")
print("Model and tokenizer successfully persisted and verified!")


Model weights saved to saved_model/transformer_weights.weights.h5
Tokenizer config saved to saved_model/tokenizer_config.json

Reload Verification:
Top 5 predicted next tokens for 'to be or not to be': ['the', 'and', 'to', 'i', 'of']
Model and tokenizer successfully persisted and verified!
